# MMLU Benchmark Claim Audit

This notebook reproduces the MMLU case study from the
Benchmark Claim Auditor project.

Research question:

> When does a higher average benchmark score actually
> provide strong evidence that one model is better than another?

The analysis evaluates magnitude, consistency, uncertainty,
aggregation sensitivity, stability, and outlier sensitivity.


In [ ]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")

from benchmark_claim_auditor import audit_pair

SEED = 42
BOOTSTRAP_RESAMPLES = 10000

np.random.seed(SEED)

print("Environment ready.")


In [ ]:
data = pd.read_csv("../data/mmlu.csv")

print("Shape:", data.shape)
print("Models:", data["Model"].nunique())

data.head()


In [ ]:
overall_column = "MMLU All Subjects - EM"

ranking = (
    data[["Model", overall_column]]
    .sort_values(overall_column, ascending=False)
    .reset_index(drop=True)
)

ranking.head(10)


In [ ]:
model_a = ranking.iloc[0]["Model"]

comparison_models = ranking.iloc[1:6]["Model"].tolist()

print("Primary model:")
print(model_a)

print("\nComparison models:")
for model in comparison_models:
    print("-", model)


In [ ]:
results = []

for model_b in comparison_models:

    result = audit_pair(
        data=data,
        model_a=model_a,
        model_b=model_b,
        overall_column=overall_column,
        bootstrap_samples=BOOTSTRAP_RESAMPLES,
        seed=SEED
    )

    results.append(result)

print("Completed", len(results), "comparisons.")


In [ ]:
summary_rows = []

for result in results:

    summary = {
        "Model B": result["model_b"],
        "A Score": result["overall_a"],
        "B Score": result["overall_b"],
        "Overall Gap": result["overall_gap"],
        "Mean Difference": result["mean_difference"],
        "Median Difference": result["median_difference"],
        "Cohen's d": result["cohens_d"],
        "Wins": result["wins"],
        "Ties": result["ties"],
        "Losses": result["losses"],
        "Win Rate": result["win_rate"],
        "Wilcoxon p": result["wilcoxon_p"],
        "Bootstrap CI Lower": result["bootstrap_ci"][0],
        "Bootstrap CI Upper": result["bootstrap_ci"][1],
        "LOO Min": result["loo_min"],
        "LOO Max": result["loo_max"],
        "LOO Range": result["loo_range"]
    }

    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)

summary_df


## Interpretation

The analysis should not treat the leaderboard gap itself as
sufficient evidence of superiority.

Instead, the headline score is examined alongside:

- task-level magnitude
- consistency
- effect size
- statistical uncertainty
- aggregation sensitivity
- leave-one-task-out stability
- outlier sensitivity

The resulting evidence profile is reported in
`../results/mmlu_claim_audit.csv`.
